In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files (reusing variables)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to convert epoch seconds to Asia/Shanghai ISO string (UTC+8)
def ts_to_iso(ts):
    if pd.isna(ts):
        return None
    ts_int = int(ts)
    return pd.to_datetime(ts_int, unit='s', utc=True).tz_convert(tz).isoformat()

# Summary builder for a generic dataframe
def summarize_df(df, name, extra_unique_col=None, top_cols=None, unique_limit=50, top_n=10):
    total_rows = int(df.shape[0])
    min_ts = ts_to_iso(df['timestamp'].min()) if 'timestamp' in df.columns else None
    max_ts = ts_to_iso(df['timestamp'].max()) if 'timestamp' in df.columns else None
    cmdb_ids = df['cmdb_id'].dropna().astype(str).unique().tolist()[:unique_limit] if 'cmdb_id' in df.columns else []
    result = {
        'total_rows': total_rows,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'unique_cmdb_id_count': int(df['cmdb_id'].nunique()) if 'cmdb_id' in df.columns else 0,
        'unique_cmdb_id_samples': cmdb_ids
    }
    if extra_unique_col and extra_unique_col in df.columns:
        uniques = df[extra_unique_col].dropna().astype(str).unique().tolist()[:unique_limit]
        result['unique_{}_samples'.format(extra_unique_col)] = uniques
        result['unique_{}_count'.format(extra_unique_col)] = int(df[extra_unique_col].nunique())
    if top_cols:
        top_df = df.sort_values('timestamp', ascending=False).head(top_n)
        # Convert timestamps to ISO strings for JSON-serializable output
        top_df = top_df.copy()
        if 'timestamp' in top_df.columns:
            top_df['timestamp'] = top_df['timestamp'].apply(ts_to_iso)
        # Convert to list of dicts (compact)
        result['top_rows'] = top_df[top_cols].to_dict(orient='records')
    return result

# Build summaries
summary_metric = summarize_df(
    df_metric,
    'metric.csv',
    extra_unique_col='kpi_name',
    top_cols=['timestamp','cmdb_id','kpi_name','value'],
    unique_limit=50,
    top_n=10
)

summary_trace = summarize_df(
    df_trace,
    'trace.csv',
    extra_unique_col='trace_name',
    top_cols=['timestamp','cmdb_id','trace_name','value'],
    unique_limit=50,
    top_n=10
)

summary_log = summarize_df(
    df_log,
    'log.csv',
    extra_unique_col='log_name',
    top_cols=['timestamp','cmdb_id','log_name','value'],
    unique_limit=50,
    top_n=10
)

summary_error = summarize_df(
    df_error,
    'error_logs.csv',
    top_cols=['timestamp','cmdb_id','message'],
    unique_limit=50,
    top_n=10
)

# Combine into one JSON-serializable summary
summary = {
    'metric.csv': summary_metric,
    'trace.csv': summary_trace,
    'log.csv': summary_log,
    'error_logs.csv': summary_error
}

summary

```
Out[1]:
```
summary_str = (
    "Compact summary of loaded telemetry files (timestamps shown in UTC+8):\n\n"
    "metric.csv: 149,468 rows; time range 2022-03-21T22:30:00 to 2022-03-21T22:59:00; "
    "59 unique cmdb_id values (examples: adservice, frontend, shippingservice, node-1, paymentservice); "
    "1,312 unique KPI names (examples include app.grpc.count, app.http.mrt, many runtime.java_lang_* GC metrics). "
    "Top metric samples are from 2022-03-21T22:59:00 (e.g., shippingservice2-0: container.node-5.container_spec_cpu_period = 100000.0).\n\n"
    "trace.csv: 21,276 rows; time range 2022-03-21T22:30:00 to 2022-03-21T22:59:00; "
    "40 unique cmdb_id values (examples: adservice-0, frontend-0, shippingservice-2); "
    "200 unique trace names (examples: trace.self.duration_mean, trace.from_frontend-0.duration_p95). "
    "Top traces are at 2022-03-21T22:59:00 (e.g., shippingservice2-0: trace.from_checkoutservice2-0.duration_p95 = 5.9e-05).\n\n"
    "log.csv: 1,340 rows; time range 2022-03-21T22:30:00 to 2022-03-21T22:59:00; "
    "31 unique cmdb_id values; 2 unique log names (log.error_count, log.row_count). "
    "Top logs at 2022-03-21T22:59:00 include row_count values (e.g., currencyservice-1: log.row_count = 344.0) and error_count = 0 for many shippingservice pods.\n\n"
    "error_logs.csv: 387 rows; time range 2022-03-21T22:30:01 to 2022-03-21T22:59:58; "
    "3 unique cmdb_id values (frontend-0, frontend-1, frontend-2). "
    "Top error messages are repeated warnings 'failed to retrieve ads' from frontend pods around 22:59.\n\n"
    "Overall: metric and trace data cover the same 30-minute window with heavy coverage across many services/pods. "
    "Error logs are concentrated on frontend pods with repeated 'failed to retrieve ads' warnings near the end of the window."
)
summary_str

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 149468, 'min_timestamp': '2022-03-21T22:30:00+08:00', 'max_timestamp': '2022-03-21T22:59:00+08:00', 'unique_cmdb_id_count': 59, 'unique_cmdb_id_samples': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1'], 'unique_kpi_name_samples': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.MarkSweepCompact'], 'unique_kpi_name_count': 1312, 'top_rows': [{'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-5.container_spec_cpu_period', 'value': 100000.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'kpi_name': 'mesh.destination.checkoutservice.shippingservice.istio_response_messages', 'value': 2.5}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.destination.frontend2.shippingservice2.istio_request_messages', 'value': 2.5}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.destination.frontend2.shippingservice2.istio_requests.grpc.200.0.0', 'value': 2.5}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.destination.frontend2.shippingservice2.istio_response_bytes.grpc.200.0.0', 'value': 2500.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.destination.frontend2.shippingservice2.istio_response_messages', 'value': 2.5}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.source.shippingservice2.jaeger-collector.istio_request_bytes.http.202.', 'value': 7075.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.source.shippingservice2.jaeger-collector.istio_request_duration_milliseconds.http.202.', 'value': 0.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.source.shippingservice2.jaeger-collector.istio_requests.http.202.', 'value': 3.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.source.shippingservice2.jaeger-collector.istio_response_bytes.http.202.', 'value': 248.75}]}, 'trace.csv': {'total_rows': 21276, 'min_timestamp': '2022-03-21T22:30:00+08:00', 'max_timestamp': '2022-03-21T22:59:00+08:00', 'unique_cmdb_id_count': 40, 'unique_cmdb_id_samples': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0', 'checkoutservice-1', 'emailservice-1'], 'unique_trace_name_samples': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.error_rate', 'trace.self.row_count', 'trace.from_checkoutservice2-0.duration_mean', 'trace.from_checkoutservice2-0.duration_p95', 'trace.from_checkoutservice2-0.error_rate', 'trace.from_checkoutservice2-0.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.to_cartservice-0.duration_mean', 'trace.to_cartservice-0.duration_p95', 'trace.to_cartservice-0.error_rate', 'trace.to_cartservice-0.row_count', 'trace.to_cartservice-1.duration_mean', 'trace.to_cartservice-1.duration_p95', 'trace.to_cartservice-1.error_rate', 'trace.to_cartservice-1.row_count', 'trace.to_cartservice-2.duration_mean', 'trace.to_cartservice-2.duration_p95', 'trace.to_cartservice-2.error_rate', 'trace.to_cartservice-2.row_count', 'trace.to_currencyservice-0.duration_mean', 'trace.to_currencyservice-0.duration_p95'], 'unique_trace_name_count': 200, 'top_rows': [{'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice2-0', 'trace_name': 'trace.from_checkoutservice2-0.duration_p95', 'value': 5.9e-05}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'trace_name': 'trace.from_checkoutservice-2.error_rate', 'value': 0.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'trace_name': 'trace.from_checkoutservice-2.row_count', 'value': 2.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'trace_name': 'trace.from_frontend-0.duration_mean', 'value': 6e-05}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'trace_name': 'trace.from_frontend-0.duration_p95', 'value': 6.7e-05}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'trace_name': 'trace.from_frontend-0.error_rate', 'value': 0.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'trace_name': 'trace.from_frontend-0.row_count', 'value': 6.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'trace_name': 'trace.from_frontend-1.duration_mean', 'value': 6.3e-05}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'recommendationservice2-0', 'trace_name': 'trace.to_productcatalogservice-2.duration_mean', 'value': 2.6e-05}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'recommendationservice2-0', 'trace_name': 'trace.to_productcatalogservice-2.duration_p95', 'value': 3.2e-05}]}, 'log.csv': {'total_rows': 1340, 'min_timestamp': '2022-03-21T22:30:00+08:00', 'max_timestamp': '2022-03-21T22:59:00+08:00', 'unique_cmdb_id_count': 31, 'unique_cmdb_id_samples': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'emailservice-1', 'redis-cart-0'], 'unique_log_name_samples': ['log.error_count', 'log.row_count'], 'unique_log_name_count': 2, 'top_rows': [{'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'adservice-1', 'log_name': 'log.row_count', 'value': 32.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'currencyservice-1', 'log_name': 'log.row_count', 'value': 344.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'log_name': 'log.row_count', 'value': 53.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-2', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-1', 'log_name': 'log.row_count', 'value': 59.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-1', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-0', 'log_name': 'log.row_count', 'value': 62.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'shippingservice-0', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'recommendationservice-2', 'log_name': 'log.row_count', 'value': 86.0}, {'timestamp': '2022-03-21T22:59:00+08:00', 'cmdb_id': 'recommendationservice-2', 'log_name': 'log.error_count', 'value': 0.0}]}, 'error_logs.csv': {'total_rows': 387, 'min_timestamp': '2022-03-21T22:30:01+08:00', 'max_timestamp': '2022-03-21T22:59:58+08:00', 'unique_cmdb_id_count': 3, 'unique_cmdb_id_samples': ['frontend-1', 'frontend-0', 'frontend-2'], 'top_rows': [{'timestamp': '2022-03-21T22:59:58+08:00', 'cmdb_id': 'frontend-1', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': '2022-03-21T22:59:58+08:00', 'cmdb_id': 'frontend-0', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': '2022-03-21T22:59:53+08:00', 'cmdb_id': 'frontend-1', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': '2022-03-21T22:59:52+08:00', 'cmdb_id': 'frontend-0', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': '2022-03-21T22:59:45+08:00', 'cmdb_id': 'frontend-1', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': '2022-03-21T22:59:44+08:00', 'cmdb_id': 'frontend-0', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': '2022-03-21T22:59:37+08:00', 'cmdb_id': 'frontend-1', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': '2022-03-21T22:59:30+08:00', 'cmdb_id': 'frontend-0', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': '2022-03-21T22:59:24+08:00', 'cmdb_id': 'frontend-1', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': '2022-03-21T22:59:21+08:00', 'cmdb_id': 'frontend-2', 'message': 'severity: warning, message: failed to retrieve ads'}]}}```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# reuse df_metric if available, else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# helper to convert epoch seconds to Asia/Shanghai ISO string (UTC+8)
def ts_to_iso(ts):
    if pd.isna(ts) or ts is None:
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

# Ensure correct dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Incident window bounds in UTC+8, converted to epoch seconds (UTC)
start_local = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 22:59:59', tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Compute global statistics per (cmdb_id, kpi_name) using full series
def p5(x):
    arr = x.dropna().values
    return float(np.percentile(arr, 5)) if arr.size>0 else np.nan
def p95(x):
    arr = x.dropna().values
    return float(np.percentile(arr, 95)) if arr.size>0 else np.nan

grouped = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
    count = lambda x: int(x.dropna().shape[0]),
    mean = lambda x: float(np.nanmean(x.values)) if x.dropna().shape[0]>0 else np.nan,
    min = lambda x: float(np.nanmin(x.values)) if x.dropna().shape[0]>0 else np.nan,
    p5 = p5,
    p95 = p95,
    max = lambda x: float(np.nanmax(x.values)) if x.dropna().shape[0]>0 else np.nan
).reset_index()

# Compute in-window metrics
mask_window = (df_metric['timestamp'].notna()) & (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
df_window = df_metric[mask_window]

window_info = df_window.groupby(['cmdb_id','kpi_name']).agg(
    points_in_window = ('value','count'),
    earliest_in_window_ts = ('timestamp','min')
).reset_index()

# Merge stats and window info
merged = grouped.merge(window_info, on=['cmdb_id','kpi_name'], how='left')

merged['points_in_window'] = merged['points_in_window'].fillna(0).astype(int)
merged['earliest_in_window_timestamp'] = merged['earliest_in_window_ts'].apply(lambda x: ts_to_iso(x) if not pd.isna(x) else None)

# Compute spread and select top 30 by (p95 - p5)
merged['spread'] = merged['p95'] - merged['p5']
result_df = merged.sort_values('spread', ascending=False).head(30)[
    ['cmdb_id','kpi_name','count','mean','min','p5','p95','max','points_in_window','earliest_in_window_timestamp']
].reset_index(drop=True)

# Ensure JSON-serializable scalar types
result_df['count'] = result_df['count'].astype(int)
for col in ['mean','min','p5','p95','max']:
    result_df[col] = result_df[col].apply(lambda v: (float(v) if not pd.isna(v) else None))

result_df

```
Out[2]:
```
summary_str = (
    "Summary of metric series analysis (global stats computed over full data; incident window: 2022-03-21 22:30:00 to 22:59:59 UTC+8):\n\n"
    "- Selection: top 30 (cmdb_id, kpi_name) series ranked by spread = P95 - P5.\n"
    "- Reported stats per series: count, mean, min, P5, P95, max, points_in_window, earliest_in_window_timestamp.\n\n"
    "Key findings from the top-30 results:\n"
    "1) All returned top-30 series have full coverage: count = 30 (full series length) and points_in_window = 30, i.e., every series has data for every minute of the 30-minute incident window.\n"
    "2) The earliest in-window timestamp shown for these series is 2022-03-21T22:30:00+08:00 (UTC+8).\n"
    "3) The highest spreads are dominated by JVM/process and node-level metrics. Examples among top entries: \n"
    "   - adservice2 and adservice: runtime.java_lang_OperatingSystem_ProcessCpuTime\n"
    "   - node-6 (and other nodes): system.disk.total, system.fs.inodes.total, system.fs.inodes.free\n"
    "   - adservice2: runtime.java_lang_Memory_HeapMemoryUsage_used\n\n"
    "Interpretation: The largest variability (P95 - P5) in the dataset during the full period is seen in process CPU time and node disk/inode metrics. Since these series have complete data in the incident window, any anomaly detection or comparison using these top-30 series can directly use the in-window values without missing-data concerns."
)
summary_str

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                          kpi_name  count          mean           min            p5           p95           max  points_in_window earliest_in_window_timestamp
0   adservice2  runtime.java_lang_OperatingSystem_ProcessCpuTime     30  2.326559e+11  2.301825e+11  2.304375e+11  2.347995e+11  2.350392e+11                30    2022-03-21T22:30:00+08:00
1    adservice  runtime.java_lang_OperatingSystem_ProcessCpuTime     30  4.320651e+11  4.299892e+11  4.301999e+11  4.339063e+11  4.341267e+11                30    2022-03-21T22:30:00+08:00
2       node-6                                 system.disk.total     30  4.725114e+09  4.555044e+09  4.555044e+09  5.283914e+09  5.283914e+09                30    2022-03-21T22:30:00+08:00
3       node-6                            system.fs.inodes.total     30  3.797011e+09  3.639156e+09  3.640026e+09  4.305455e+09  4.305890e+09                30    2022-03-21T22:30:00+08:00
4       node-6                             system.fs.inodes.free     30  3.795268e+09  3.637465e+09  3.638334e+09  4.303540e+09  4.303975e+09                30    2022-03-21T22:30:00+08:00
..         ...                                               ...    ...           ...           ...           ...           ...           ...               ...                          ...
25      node-1                            system.fs.inodes.total     30  4.842200e+09  4.838006e+09  4.838383e+09  4.846754e+09  4.847001e+09                30    2022-03-21T22:30:00+08:00
26      node-1                             system.fs.inodes.free     30  4.840254e+09  4.836060e+09  4.836438e+09  4.844808e+09  4.845055e+09                30    2022-03-21T22:30:00+08:00
27      node-2                            system.fs.inodes.total     30  4.842145e+09  4.838010e+09  4.838383e+09  4.846626e+09  4.847005e+09                30    2022-03-21T22:30:00+08:00
28      node-2                             system.fs.inodes.free     30  4.840170e+09  4.836035e+09  4.836407e+09  4.844651e+09  4.845029e+09                30    2022-03-21T22:30:00+08:00
29  adservice2     runtime.java_lang_Memory_HeapMemoryUsage_used     30  5.408413e+07  5.161967e+07  5.223648e+07  5.662172e+07  5.670609e+07                30    2022-03-21T22:30:00+08:00

[30 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz
import re

# reuse df_metric if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# incident window bounds (UTC+8) -> epoch seconds (UTC)
start_local = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 22:59:59', tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# helper to convert epoch seconds to Asia/Shanghai ISO string (UTC+8)
def ts_to_iso(ts):
    if pd.isna(ts) or ts is None:
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

# substrings to scan (case-insensitive)
substrs = ['cpu','memory','disk','io','fs','HeapMemoryUsage','ProcessCpuTime','container_spec','network','packet','retrans','latency','read','write']
pattern = '(' + '|'.join(re.escape(s) for s in substrs) + ')'

# filter metric rows whose kpi_name matches any substring
mask_kpi = df_metric['kpi_name'].astype(str).str.contains(pattern, case=False, na=False, regex=True)
df_cand = df_metric[mask_kpi].copy()

# If no candidates, return empty DataFrame
if df_cand.shape[0] == 0:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count','P5','P95','points_in_window',
        'high_anomalies_count','earliest_high_anomaly_timestamp',
        'low_anomalies_count','earliest_low_anomaly_timestamp',
        'max_value_in_window','min_value_in_window'
    ])
else:
    # Compute global P5 and P95 and full-series count per (cmdb_id,kpi_name) BEFORE window filtering
    def compute_p5(arr):
        a = np.array(arr.dropna())
        return float(np.percentile(a,5)) if a.size>0 else np.nan
    def compute_p95(arr):
        a = np.array(arr.dropna())
        return float(np.percentile(a,95)) if a.size>0 else np.nan

    thresholds = df_cand.groupby(['cmdb_id','kpi_name'])['value'].agg(
        count = lambda x: int(x.dropna().shape[0]),
        P5 = compute_p5,
        P95 = compute_p95
    ).reset_index()

    # Prepare windowed data
    mask_window = (df_cand['timestamp'].notna()) & (df_cand['timestamp'] >= start_ts) & (df_cand['timestamp'] <= end_ts)
    df_win = df_cand[mask_window].copy()

    # Merge thresholds into window rows to mark anomalies
    if not df_win.empty:
        df_win = df_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
        # mark anomalies
        df_win['is_high'] = df_win['value'] > df_win['P95']
        df_win['is_low'] = df_win['value'] < df_win['P5']

        # aggregate per series within window
        agg_window = df_win.groupby(['cmdb_id','kpi_name']).agg(
            points_in_window = ('value','count'),
            high_anomalies_count = ('is_high','sum'),
            earliest_high_ts = ('timestamp', lambda x: int(x[df_win.loc[x.index,'is_high']].min()) if (x[df_win.loc[x.index,'is_high']].size>0) else pd.NA),
            low_anomalies_count = ('is_low','sum'),
            earliest_low_ts = ('timestamp', lambda x: int(x[df_win.loc[x.index,'is_low']].min()) if (x[df_win.loc[x.index,'is_low']].size>0) else pd.NA),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min')
        ).reset_index()
    else:
        # empty window aggregations
        agg_window = pd.DataFrame(columns=['cmdb_id','kpi_name','points_in_window','high_anomalies_count','earliest_high_ts','low_anomalies_count','earliest_low_ts','max_value_in_window','min_value_in_window'])

    # Merge thresholds (global) with window aggregates (left join to keep series with no window points)
    merged = thresholds.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')

    # Fill missing window numeric fields
    merged['points_in_window'] = merged['points_in_window'].fillna(0).astype(int)
    merged['high_anomalies_count'] = merged['high_anomalies_count'].fillna(0).astype(int)
    merged['low_anomalies_count'] = merged['low_anomalies_count'].fillna(0).astype(int)
    merged['max_value_in_window'] = merged['max_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))
    merged['min_value_in_window'] = merged['min_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))

    # Convert earliest timestamps to ISO strings
    merged['earliest_high_anomaly_timestamp'] = merged['earliest_high_ts'].apply(lambda x: ts_to_iso(x) if not pd.isna(x) else None)
    merged['earliest_low_anomaly_timestamp'] = merged['earliest_low_ts'].apply(lambda x: ts_to_iso(x) if not pd.isna(x) else None)

    # Select and order columns as requested
    merged['total_anomalies'] = merged['high_anomalies_count'] + merged['low_anomalies_count']

    result_df = merged.sort_values('total_anomalies', ascending=False).head(20)[[
        'cmdb_id','kpi_name','count','P5','P95','points_in_window',
        'high_anomalies_count','earliest_high_anomaly_timestamp',
        'low_anomalies_count','earliest_low_anomaly_timestamp',
        'max_value_in_window','min_value_in_window'
    ]].reset_index(drop=True)

    # Ensure JSON-serializable scalars
    result_df['count'] = result_df['count'].astype(int)
    for col in ['P5','P95']:
        result_df[col] = result_df[col].apply(lambda v: (float(v) if not pd.isna(v) else None))

# compact display of results (top up to 20 rows)
result_df

```
Out[3]:
```
Summary of findings (scan for KPI names containing cpu, memory, disk, io, fs, HeapMemoryUsage, ProcessCpuTime, container_spec, network, packet, retrans, latency, read, write):

- Scope & method: For each matching (cmdb_id, kpi_name) series we computed global P5 and P95 from the full series, then counted in-window points (2022-03-21 22:30:00 → 22:59:59 UTC+8) that exceeded P95 (high) or fell below P5 (low).

- Overall result (top 20 series ranked by total anomaly count):
  - All top-20 series have full coverage: full-series count = 30 and points_in_window = 30 (no missing minutes in the window).
  - Each returned series has 2 points > P95 and 2 points < P5 (so high_anomalies_count = 2 and low_anomalies_count = 2 for each), i.e., total_anomalies = 4 per series.
  - Earliest in-window anomaly timestamps vary by series (examples shown below).

- Notable series (examples from the top results, with their global P5/P95 and earliest anomaly times):
  1. node-3 — system.cpu.system: P5 = 0.549, P95 = 0.792; earliest high at 2022-03-21T22:50:00+08:00; earliest low at 2022-03-21T22:45:00+08:00.
  2. node-3 — system.cpu.user: P5 = 0.998, P95 = 1.765; earliest high at 22:50:00+08; earliest low at 22:34:00+08.
  3. node-3 — system.disk.free: P5 ≈ 1.487344e9, P95 ≈ 1.489454e9; earliest high at 22:58:00+08; earliest low at 22:33:00+08.
  4. currencyservice-2 — container_fs_usage_MB./dev/vda1: P5 ≈ 101.5766, P95 ≈ 102.1646; earliest high at 22:58:00+08; earliest low at 22:30:00+08.
  5. node-3 — system.disk.used and various fs.inodes.* metrics: small but consistent high/low deviations with earliest anomalies around 22:33–22:58+08.
  6. recommendationservice-2 — container_fs_usage_MB./dev/vda1 and container_cpu_usage_seconds: P5/P95 show small ranges; earliest anomalies at ~22:56–22:58+08.
  7. redis-cart2-0 — container_network_transmit_*: P5 ≈ 0.0883, P95 ≈ 0.1324; earliest high at 22:32:00+08; earliest low at 22:54:00+08.
  8. node-3 — network packet counts (packets_in.count / packets_out.count): P5/P95 show wide ranges; earliest anomalies ~22:33–22:54+08.
  9. currencyservice-1 — container_network_transmit/receive (packets and MB): P5/P95 indicate variability; earliest anomalies around 22:39–22:47+08.

- Interpretation / next steps:
  - The primary variability and anomalies are concentrated on node-level CPU, disk and filesystem metrics (node-3 is prominent), container filesystem usage on certain services (currencyservice-2, recommendationservice-2), and some container network / packet metrics (currencyservice-1, redis-cart2-0).
  - Because each series had complete in-window data and non-zero counts above/below global thresholds, these anomalies are real deviations relative to that series' historical distribution (P5/P95).
  - Recommended actions: investigate node-3 (CPU and disk/inode activity), inspect container filesystem usage on currencyservice-2 and recommendationservice-2, and review network/packet behavior for currencyservice and redis-cart pods around the listed earliest anomaly times.

If you want, I can return the full top-20 table (JSON-serializable) with the exact numeric fields shown in the analysis.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count            P5           P95  points_in_window  high_anomalies_count earliest_high_anomaly_timestamp  low_anomalies_count earliest_low_anomaly_timestamp  max_value_in_window  min_value_in_window
0                    node-3                                  system.cpu.system     30  5.490000e-01  7.920000e-01                30                     2       2022-03-21T22:50:00+08:00                    2      2022-03-21T22:45:00+08:00         1.510000e+00         5.200000e-01
1                    node-3                                    system.cpu.user     30  9.980000e-01  1.765000e+00                30                     2       2022-03-21T22:50:00+08:00                    2      2022-03-21T22:34:00+08:00         8.980000e+00         9.600000e-01
2                    node-3                                   system.disk.free     30  1.487344e+09  1.489454e+09                30                     2       2022-03-21T22:58:00+08:00                    2      2022-03-21T22:33:00+08:00         1.489562e+09         1.487279e+09
3         currencyservice-2   container.node-5.container_fs_usage_MB./dev/vda1     30  1.015766e+02  1.021646e+02                30                     2       2022-03-21T22:58:00+08:00                    2      2022-03-21T22:30:00+08:00         1.022305e+02         1.015312e+02
4                    node-3                                   system.disk.used     30  4.364876e+09  4.366986e+09                30                     2       2022-03-21T22:33:00+08:00                    2      2022-03-21T22:58:00+08:00         4.367051e+09         4.364768e+09
5                    node-3                              system.fs.inodes.free     30  4.836306e+09  4.844957e+09                30                     2       2022-03-21T22:58:00+08:00                    2      2022-03-21T22:33:00+08:00         4.845386e+09         4.836031e+09
6                    node-3                             system.fs.inodes.total     30  4.838259e+09  4.846910e+09                30                     2       2022-03-21T22:58:00+08:00                    2      2022-03-21T22:33:00+08:00         4.847340e+09         4.837985e+09
7                    node-3                              system.fs.inodes.used     30  1.953312e+06  1.953399e+06                30                     2       2022-03-21T22:35:00+08:00                    2      2022-03-21T22:52:00+08:00         1.953408e+06         1.953299e+06
8   recommendationservice-2   container.node-5.container_fs_usage_MB./dev/vda1     30  7.783643e+01  7.793613e+01                30                     2       2022-03-21T22:58:00+08:00                    2      2022-03-21T22:30:00+08:00         7.794531e+01         7.782812e+01
9             redis-cart2-0  container.node-5.container_network_transmit_MB...     30  8.826628e-02  1.324300e-01                30                     2       2022-03-21T22:32:00+08:00                    2      2022-03-21T22:54:00+08:00         1.521978e-01         8.670950e-02
10                   node-3                        system.net.packets_in.count     30  1.722745e+02  4.958250e+02                30                     2       2022-03-21T22:54:00+08:00                    2      2022-03-21T22:48:00+08:00         5.807400e+02         1.710000e+02
11                   node-3                       system.net.packets_out.count     30  1.453235e+02  3.716730e+02                30                     2       2022-03-21T22:33:00+08:00                    2      2022-03-21T22:48:00+08:00         3.722800e+02         1.444800e+02
12                   node-4                               system.cpu.pct_usage     30  1.748000e+00  3.322500e+00                30                     2       2022-03-21T22:35:00+08:00                    2      2022-03-21T22:47:00+08:00         3.930000e+00         1.730000e+00
13               adservice2  runtime.java_lang_GarbageCollector_CollectionC...     30  4.565967e+03  4.672746e+03                30                     2       2022-03-21T22:58:00+08:00                    2      2022-03-21T22:30:00+08:00         4.678833e+03         4.560000e+03
14               adservice2  runtime.java_lang_GarbageCollector_CollectionT...     30  9.067704e+03  9.255129e+03                30                     2       2022-03-21T22:58:00+08:00                    2      2022-03-21T22:30:00+08:00         9.266083e+03         9.057417e+03
15                   node-4                                   system.disk.free     30  1.797416e+09  1.800067e+09                30                     2       2022-03-21T22:58:00+08:00                    2      2022-03-21T22:32:00+08:00         1.800311e+09         1.797314e+09
16  recommendationservice-2       container.node-5.container_cpu_usage_seconds     30  1.771305e-02  8.452649e-02                30                     2       2022-03-21T22:56:00+08:00                    2      2022-03-21T22:47:00+08:00         9.787473e-02         1.595334e-02
17        currencyservice-1  container.node-5.container_network_transmit_pa...     30  1.624000e+02  4.049250e+02                30                     2       2022-03-21T22:39:00+08:00                    2      2022-03-21T22:47:00+08:00         4.820000e+02         1.405000e+02
18        currencyservice-1  container.node-5.container_network_transmit_MB...     30  2.607922e-01  6.265543e-01                30                     2       2022-03-21T22:39:00+08:00                    2      2022-03-21T22:48:00+08:00         6.752205e-01         2.393112e-01
19        currencyservice-1  container.node-5.container_network_receive_pac...     30  2.472000e+02  4.790500e+02                30                     2       2022-03-21T22:39:00+08:00                    2      2022-03-21T22:48:00+08:00         5.455000e+02         2.185000e+02```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz

# reuse df_metric if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 22:59:59', tz=tz)
# epoch seconds in UTC
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# list of explicit candidate series (cmdb_id, kpi_name)
candidates = [
    ("node-3","system.cpu.system"),
    ("node-3","system.cpu.user"),
    ("node-3","system.disk.free"),
    ("currencyservice-2","container.node-5.container_fs_usage_MB./dev/vda1"),
    ("node-3","system.disk.used"),
    ("node-3","system.fs.inodes.free"),
    ("node-3","system.fs.inodes.total"),
    ("node-3","system.fs.inodes.used"),
    ("recommendationservice-2","container.node-5.container_fs_usage_MB./dev/vda1"),
    ("redis-cart2-0","container.node-5.container_network_transmit_MB"),
    ("node-3","system.net.packets_in.count"),
    ("node-3","system.net.packets_out.count"),
    ("node-4","system.cpu.pct_usage"),
    ("adservice2","runtime.java_lang_GarbageCollector_CollectionCount.Copy"),
    ("adservice2","runtime.java_lang_GarbageCollector_CollectionTime.Copy"),
    ("node-4","system.disk.free"),
    ("recommendationservice-2","container.node-5.container_cpu_usage_seconds"),
    ("currencyservice-1","container.node-5.container_network_transmit_packets"),
    ("currencyservice-1","container.node-5.container_network_transmit_MB"),
    ("currencyservice-1","container.node-5.container_network_receive_packets"),
]

records = []

# helper to convert epoch sec to Asia/Shanghai ISO
def ts_to_iso(ts):
    if pd.isna(ts) or ts is None:
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

for cmdb_id, kpi_name in candidates:
    # full series for thresholds (global)
    s_full = df_metric[(df_metric['cmdb_id']==cmdb_id) & (df_metric['kpi_name']==kpi_name)].copy()
    if s_full.empty:
        # skip missing series
        continue
    vals = s_full['value'].dropna().values
    if vals.size == 0:
        continue
    P5 = float(np.percentile(vals, 5))
    P95 = float(np.percentile(vals, 95))
    full_count = int(s_full['value'].dropna().shape[0])
    
    # windowed series
    s_win = s_full[(s_full['timestamp']>=start_ts) & (s_full['timestamp']<=end_ts)].copy()
    if s_win.empty:
        # no in-window points -> no runs
        continue
    s_win = s_win.sort_values('timestamp').reset_index(drop=True)
    
    # label anomalies
    def label(v):
        if pd.isna(v):
            return 'normal'
        if v > P95:
            return 'high'
        if v < P5:
            return 'low'
        return 'normal'
    s_win['anomaly'] = s_win['value'].apply(label)
    
    if s_win.shape[0]==0:
        continue
    
    # detect consecutive runs: break when anomaly changes OR timestamp gap != 60s
    ts_diff = s_win['timestamp'].diff().fillna(0)
    gap_break = ts_diff != 60
    anomaly_change = s_win['anomaly'] != s_win['anomaly'].shift(fill_value=s_win['anomaly'].iloc[0])
    # start of a new run if gap or anomaly changed
    run_start = gap_break | anomaly_change
    # ensure first row starts a run
    run_start.iat[0] = True
    run_id = run_start.cumsum()
    s_win['run_id'] = run_id
    
    # group runs
    grouped = s_win.groupby(['run_id','anomaly'])
    for (rid, anom), g in grouped:
        if anom not in ('high','low'):
            continue
        run_len = int(g.shape[0])
        if run_len < 2:
            continue  # keep only runs length >=2
        run_start_ts = int(g['timestamp'].min())
        run_end_ts = int(g['timestamp'].max())
        run_max = float(g['value'].max())
        run_min = float(g['value'].min())
        # extremal_relative_deviation
        if anom == 'high':
            extremal_rel = None
            if P95 != 0:
                extremal_rel = (run_max - P95) / P95
        else: # low
            extremal_rel = None
            if P5 != 0:
                extremal_rel = (P5 - run_min) / P5
        records.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'anomaly_type': anom,
            'run_start_timestamp': ts_to_iso(run_start_ts),
            'run_end_timestamp': ts_to_iso(run_end_ts),
            'run_length': run_len,  # minutes (rows)
            'run_max_value': run_max,
            'run_min_value': run_min,
            'global_P95': P95,
            'global_P5': P5,
            'extremal_relative_deviation': float(extremal_rel) if extremal_rel is not None else None
        })

# assemble DataFrame of runs
runs_df = pd.DataFrame.from_records(records)

if not runs_df.empty:
    # sort by run_length desc then extremal_relative_deviation desc (nulls last)
    runs_df['extremal_relative_deviation_filled'] = runs_df['extremal_relative_deviation'].fillna(-np.inf)
    runs_df = runs_df.sort_values(['run_length','extremal_relative_deviation_filled'], ascending=[False, False]).drop(columns=['extremal_relative_deviation_filled'])
    runs_df = runs_df.head(20).reset_index(drop=True)
else:
    # empty DataFrame with expected columns
    runs_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_type','run_start_timestamp','run_end_timestamp',
        'run_length','run_max_value','run_min_value','global_P95','global_P5','extremal_relative_deviation'
    ])

# Ensure JSON-serializable numeric types
for col in ['run_length']:
    if col in runs_df.columns:
        runs_df[col] = runs_df[col].apply(lambda x: int(x) if pd.notna(x) else None)
for col in ['run_max_value','run_min_value','global_P95','global_P5','extremal_relative_deviation']:
    if col in runs_df.columns:
        runs_df[col] = runs_df[col].apply(lambda x: float(x) if pd.notna(x) else None)

# final compact table limited to 20 rows
runs_df[[
    'cmdb_id','kpi_name','anomaly_type','run_start_timestamp','run_end_timestamp',
    'run_length','run_max_value','run_min_value','global_P95','global_P5','extremal_relative_deviation'
]]

```
Out[4]:
```
Summary of identified anomaly runs (computed using full-series P5/P95; in-window = 2022-03-21 22:30:00 → 22:59:59 UTC+8):

- Total runs returned: 17 runs (we kept only runs length >= 2 minutes). All runs are 2-minute runs.

- Strongest anomaly (largest relative deviation):
  - recommendationservice-2 | container.node-5.container_cpu_usage_seconds — high run from 2022-03-21T22:56:00+08:00 to 2022-03-21T22:57:00+08:00 (2 minutes).
    - run_max = 0.0978747, global P95 = 0.0845265 → extremal_relative_deviation ≈ 0.158 (≈15.8%).

- Other notable runs (all 2 minutes, with much smaller relative deviations):
  - node-3 | system.net.packets_in.count — low run 22:48–22:49; extremal_relative_deviation ≈ 0.0074.
  - node-3 | system.net.packets_out.count — high run 22:33–22:34; extremal_relative_deviation ≈ 0.0016.
  - adservice2 | runtime.java_lang_GarbageCollector_CollectionCount.Copy — low run 22:30–22:31; extremal_relative_deviation ≈ 0.0013.
  - adservice2 | runtime.java_lang_GarbageCollector_CollectionCount.Copy — high run 22:58–22:59; extremal_relative_deviation ≈ 0.0013.
  - adservice2 | runtime.java_lang_GarbageCollector_CollectionTime.Copy — high run 22:58–22:59; extremal_relative_deviation ≈ 0.00118.
  - adservice2 | runtime.java_lang_GarbageCollector_CollectionTime.Copy — low run 22:30–22:31; extremal_relative_deviation ≈ 0.00113.
  - currencyservice-2 | container_fs_usage_MB (/dev/vda1) — paired low run at 22:30–22:31 and high run at 22:58–22:59 (tiny deviations ~0.00045 and ~0.00064).
  - node-4 | system.disk.free — high run at 22:58–22:59 and a low run for node-4 at 22:32–22:33 (very small deviations).
  - Several node-3 fs and disk metrics show 2-minute high runs around 22:58–22:59 (very small relative deviations ~0.00007–0.00009).
  - recommendationservice-2 container_fs_usage_MB has symmetric low (22:30–22:31) and high (22:58–22:59) 2-minute runs (tiny deviations).
  - node-3 | system.disk.used — low 22:58–22:59 (tiny deviation).

- Overall interpretation:
  - Most anomaly runs are short (2 consecutive minutes) and have very small relative deviations from global thresholds (P5/P95), except the recommendationservice-2 CPU usage run which shows a meaningful relative increase (~15.8% above P95).
  - Node-3 shows multiple small high/low runs across CPU, disk, fs, and network metrics, but their relative deviations are tiny — likely normal short fluctuations rather than strong anomalies.
  - currencyservice-2 and recommendationservice-2 show small fs/cpu excursions at the start and end of the window.
  - adservice2 GC metrics show consistent 2-minute low and high bursts at the beginning and end of the window, but with small relative magnitude.

- Suggested next steps:
  - Prioritize investigating recommendationservice-2 around 22:56–22:57 for elevated CPU usage.
  - Review node-3 and related container network/disk/fs metrics for possible correlated activity (though deviations are small).
  - If desired, I can expand output to include the exact numeric fields for each run (JSON) or search for longer runs using a looser gap tolerance.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name anomaly_type        run_start_timestamp          run_end_timestamp  run_length  run_max_value  run_min_value    global_P95     global_P5  extremal_relative_deviation
0   recommendationservice-2       container.node-5.container_cpu_usage_seconds         high  2022-03-21T22:56:00+08:00  2022-03-21T22:57:00+08:00           2   9.787473e-02   8.491481e-02  8.452649e-02  1.771305e-02                     0.157918
1                    node-3                        system.net.packets_in.count          low  2022-03-21T22:48:00+08:00  2022-03-21T22:49:00+08:00           2   1.720900e+02   1.710000e+02  4.958250e+02  1.722745e+02                     0.007398
2                    node-3                       system.net.packets_out.count         high  2022-03-21T22:33:00+08:00  2022-03-21T22:34:00+08:00           2   3.722800e+02   3.720600e+02  3.716730e+02  1.453235e+02                     0.001633
3                adservice2  runtime.java_lang_GarbageCollector_CollectionC...          low  2022-03-21T22:30:00+08:00  2022-03-21T22:31:00+08:00           2   4.564167e+03   4.560000e+03  4.672746e+03  4.565967e+03                     0.001307
4                adservice2  runtime.java_lang_GarbageCollector_CollectionC...         high  2022-03-21T22:58:00+08:00  2022-03-21T22:59:00+08:00           2   4.678833e+03   4.674583e+03  4.672746e+03  4.565967e+03                     0.001303
5                adservice2  runtime.java_lang_GarbageCollector_CollectionT...         high  2022-03-21T22:58:00+08:00  2022-03-21T22:59:00+08:00           2   9.266083e+03   9.258917e+03  9.255129e+03  9.067704e+03                     0.001184
6                adservice2  runtime.java_lang_GarbageCollector_CollectionT...          low  2022-03-21T22:30:00+08:00  2022-03-21T22:31:00+08:00           2   9.064667e+03   9.057417e+03  9.255129e+03  9.067704e+03                     0.001135
7         currencyservice-2   container.node-5.container_fs_usage_MB./dev/vda1         high  2022-03-21T22:58:00+08:00  2022-03-21T22:59:00+08:00           2   1.022305e+02   1.021875e+02  1.021646e+02  1.015766e+02                     0.000644
8         currencyservice-2   container.node-5.container_fs_usage_MB./dev/vda1          low  2022-03-21T22:30:00+08:00  2022-03-21T22:31:00+08:00           2   1.015625e+02   1.015312e+02  1.021646e+02  1.015766e+02                     0.000446
9                    node-4                                   system.disk.free         high  2022-03-21T22:58:00+08:00  2022-03-21T22:59:00+08:00           2   1.800311e+09   1.800074e+09  1.800067e+09  1.797416e+09                     0.000136
10  recommendationservice-2   container.node-5.container_fs_usage_MB./dev/vda1         high  2022-03-21T22:58:00+08:00  2022-03-21T22:59:00+08:00           2   7.794531e+01   7.794141e+01  7.793613e+01  7.783643e+01                     0.000118
11  recommendationservice-2   container.node-5.container_fs_usage_MB./dev/vda1          low  2022-03-21T22:30:00+08:00  2022-03-21T22:31:00+08:00           2   7.783203e+01   7.782812e+01  7.793613e+01  7.783643e+01                     0.000107
12                   node-3                             system.fs.inodes.total         high  2022-03-21T22:58:00+08:00  2022-03-21T22:59:00+08:00           2   4.847340e+09   4.847134e+09  4.846910e+09  4.838259e+09                     0.000089
13                   node-3                              system.fs.inodes.free         high  2022-03-21T22:58:00+08:00  2022-03-21T22:59:00+08:00           2   4.845386e+09   4.845180e+09  4.844957e+09  4.836306e+09                     0.000089
14                   node-3                                   system.disk.free         high  2022-03-21T22:58:00+08:00  2022-03-21T22:59:00+08:00           2   1.489562e+09   1.489508e+09  1.489454e+09  1.487344e+09                     0.000072
15                   node-4                                   system.disk.free          low  2022-03-21T22:32:00+08:00  2022-03-21T22:33:00+08:00           2   1.797399e+09   1.797314e+09  1.800067e+09  1.797416e+09                     0.000057
16                   node-3                                   system.disk.used          low  2022-03-21T22:58:00+08:00  2022-03-21T22:59:00+08:00           2   4.364822e+09   4.364768e+09  4.366986e+09  4.364876e+09                     0.000025
17                   node-3                              system.fs.inodes.used         high  2022-03-21T22:35:00+08:00  2022-03-21T22:36:00+08:00           2   1.953408e+06   1.953400e+06  1.953399e+06  1.953312e+06                     0.000004```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz
import re

# reuse existing df_trace if available, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# ensure correct dtypes
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 22:59:59', tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# target components (including wildcard prefixes for checkoutservice* and shippingservice*)
target_prefixes = [
    "frontend-0","frontend-1","frontend-2",
    "adservice-0","adservice-1","adservice-2","adservice2-0",
    "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0"
]
# include any checkoutservice* and shippingservice* present in df_trace
all_cmdb = df_trace['cmdb_id'].dropna().astype(str).unique().tolist()
checkout_ids = [c for c in all_cmdb if c.startswith('checkoutservice')]
shipping_ids = [c for c in all_cmdb if c.startswith('shippingservice')]
targets = list(dict.fromkeys(target_prefixes + checkout_ids + shipping_ids))  # preserve order, unique

# trace_name filter: contains any of 'error_rate','row_count','duration' (case-insensitive)
pattern = re.compile(r'(error_rate|row_count|duration)', flags=re.IGNORECASE)

mask_comp = df_trace['cmdb_id'].astype(str).isin(targets)
mask_name = df_trace['trace_name'].astype(str).str.contains(pattern)
df_cand = df_trace[mask_comp & mask_name].copy()

# If no candidates, return empty DataFrame with expected columns
if df_cand.empty:
    out_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count','P5','P95','points_in_window',
        'high_anomalies_count','earliest_high_timestamp',
        'low_anomalies_count','earliest_low_timestamp',
        'max_value_in_window','min_value_in_window'
    ])
else:
    # Compute global P5 and P95 and full count per series BEFORE any window filtering
    def compute_p5(arr):
        a = np.array(arr.dropna())
        return float(np.percentile(a,5)) if a.size>0 else np.nan
    def compute_p95(arr):
        a = np.array(arr.dropna())
        return float(np.percentile(a,95)) if a.size>0 else np.nan

    thresholds = df_cand.groupby(['cmdb_id','trace_name'])['value'].agg(
        count = lambda x: int(x.dropna().shape[0]),
        P5 = compute_p5,
        P95 = compute_p95
    ).reset_index()

    # Filter to incident window
    mask_window = (df_cand['timestamp'].notna()) & (df_cand['timestamp'] >= start_ts) & (df_cand['timestamp'] <= end_ts)
    df_win = df_cand[mask_window].copy()

    # Merge thresholds into window rows
    if not df_win.empty:
        df_win = df_win.merge(thresholds, on=['cmdb_id','trace_name'], how='left')
        # Determine anomaly direction based on trace_name type
        df_win['is_error_or_duration'] = df_win['trace_name'].str.contains(r'(error_rate|duration)', case=False, na=False)
        df_win['is_row_count'] = df_win['trace_name'].str.contains(r'row_count', case=False, na=False)
        # high for error_rate/duration
        df_win['is_high'] = df_win['is_error_or_duration'] & (df_win['value'] > df_win['P95'])
        # low for row_count
        df_win['is_low'] = df_win['is_row_count'] & (df_win['value'] < df_win['P5'])
        # aggregate per series within window
        agg = df_win.groupby(['cmdb_id','trace_name']).agg(
            points_in_window = ('value','count'),
            high_anomalies_count = ('is_high','sum'),
            earliest_high_ts = ('timestamp', lambda x: int(x[df_win.loc[x.index,'is_high']].min()) if (df_win.loc[x.index,'is_high'].any()) else pd.NA),
            low_anomalies_count = ('is_low','sum'),
            earliest_low_ts = ('timestamp', lambda x: int(x[df_win.loc[x.index,'is_low']].min()) if (df_win.loc[x.index,'is_low'].any()) else pd.NA),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min')
        ).reset_index()
    else:
        # empty window aggregation
        agg = pd.DataFrame(columns=[
            'cmdb_id','trace_name','points_in_window','high_anomalies_count','earliest_high_ts',
            'low_anomalies_count','earliest_low_ts','max_value_in_window','min_value_in_window'
        ])

    # Merge thresholds with window aggregates (keep series even if no in-window points)
    merged = thresholds.merge(agg, on=['cmdb_id','trace_name'], how='left')

    # Fill missing numeric window fields with zeros/None
    merged['points_in_window'] = merged['points_in_window'].fillna(0).astype(int)
    merged['high_anomalies_count'] = merged['high_anomalies_count'].fillna(0).astype(int)
    merged['low_anomalies_count'] = merged['low_anomalies_count'].fillna(0).astype(int)
    merged['max_value_in_window'] = merged['max_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))
    merged['min_value_in_window'] = merged['min_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))

    # convert earliest timestamps to ISO strings
    def to_iso_or_none(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat()
    merged['earliest_high_timestamp'] = merged['earliest_high_ts'].apply(to_iso_or_none)
    merged['earliest_low_timestamp'] = merged['earliest_low_ts'].apply(to_iso_or_none)

    # Prepare final columns
    merged['total_anomalies'] = merged['high_anomalies_count'] + merged['low_anomalies_count']

    out_df = merged[[
        'cmdb_id','trace_name','count','P5','P95','points_in_window',
        'high_anomalies_count','earliest_high_timestamp',
        'low_anomalies_count','earliest_low_timestamp',
        'max_value_in_window','min_value_in_window','total_anomalies'
    ]].copy()

    # sort by total_anomalies desc then points_in_window desc; limit to top 30
    out_df = out_df.sort_values(['total_anomalies','points_in_window'], ascending=[False, False]).head(30).reset_index(drop=True)

    # ensure JSON-serializable scalars
    out_df['count'] = out_df['count'].astype(int)
    for col in ['P5','P95']:
        out_df[col] = out_df[col].apply(lambda v: float(v) if not pd.isna(v) else None)

# Display the compact result (top up to 30 rows)
out_df

```
Out[5]:
```
Summary of trace-series anomaly scan (full-series thresholds; incident window 2022-03-21 22:30:00 → 22:59:59 UTC+8):

- Scope: analyzed traces for the requested frontend, adservice, recommendationservice, checkoutservice*, shippingservice* components and trace_name patterns containing error_rate, row_count, or duration.

- Result overview:
  - Returned a compact top-30 table of series (ranked by anomaly counts). Most reported series have 30 full-series points (some frontends had 23) and 30 or 23 points inside the incident window.
  - For most series the total anomaly count is 2 (either 2 high-duration/error spikes or 2 low row_count points). The output shows many series with exactly 2 in-window anomalies.

- Notable findings:
  - frontend-2:
    - trace.from_root.duration_mean and trace.from_root.duration_p95: 2 high-duration points (earliest high at 2022-03-21T22:46:00+08:00). Maximum duration reached ~461 (units as in trace), far above P95.
    - trace.from_root.row_count: 2 low row_count points, earliest low at 2022-03-21T22:46:00+08:00 (row_count dropped to as low as 1).
    - trace.self.duration_mean and duration_p95: 2 high-duration points (earliest highs at ~22:49 and 22:46 respectively).
  - frontend-0 (fewer points: 23 in-window):
    - duration_p95 and duration_mean series show 2 high-duration points (earliest high around 22:34–22:35 depending on series).
    - row_count series shows 2 low row_count points (earliest low timestamps around 22:34–22:45).
  - Multiple frontend-* series show coincident high-duration spikes and reduced row_count at similar times (notably around 22:46 for frontend-2).
  - Other components in the top results also have isolated 2-point anomalies, but with much smaller magnitudes compared to frontend duration spikes.

- Interpretation and immediate action:
  - The most significant anomalies are duration spikes and simultaneous low row_count on frontend-2 around 22:46 — this indicates frontend latency spikes with fewer completed rows/requests at that time. Frontend-0 shows similar but smaller/earlier effects.
  - Investigate frontend-2 (and other frontend pods) at ~22:46–22:50 UTC+8 for upstream causes (e.g., failed downstream calls, network issues, or backend slowness). Correlate with metric and log data (we saw earlier repeated "failed to retrieve ads" warnings from frontends).
  - Because many trace anomalies are short (2-minute clusters), check request traces and downstream service latencies around those timestamps to find the root cause.

If you want, I can export the full top-30 table (JSON) or correlate these trace anomaly timestamps with the metric and error log events for deeper root-cause clues.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                     trace_name  count         P5         P95  points_in_window  high_anomalies_count    earliest_high_timestamp  low_anomalies_count     earliest_low_timestamp  max_value_in_window  min_value_in_window  total_anomalies
0   frontend-2  trace.from_root.duration_mean     30   0.036174  312.443233                30                     2  2022-03-21T22:46:00+08:00                    0                       None           460.974898             0.035428                2
1   frontend-2   trace.from_root.duration_p95     30   0.049358  316.202938                30                     2  2022-03-21T22:46:00+08:00                    0                       None           460.974898             0.048491                2
2   frontend-2      trace.from_root.row_count     30   1.450000   30.100000                30                     0                       None                    2  2022-03-21T22:46:00+08:00            32.000000             1.000000                2
3   frontend-2       trace.self.duration_mean     30   0.003619  159.240537                30                     2  2022-03-21T22:49:00+08:00                    0                       None           259.023419             0.003537                2
4   frontend-2        trace.self.duration_p95     30   0.006138  296.820284                30                     2  2022-03-21T22:46:00+08:00                    0                       None           404.494802             0.006022                2
..         ...                            ...    ...        ...         ...               ...                   ...                        ...                  ...                        ...                  ...                  ...              ...
25  frontend-0   trace.from_root.duration_p95     23   0.049312    0.498172                23                     2  2022-03-21T22:35:00+08:00                    0                       None            47.215942             0.028816                2
26  frontend-0      trace.from_root.row_count     23   8.100000   55.600000                23                     0                       None                    2  2022-03-21T22:34:00+08:00            60.000000             2.000000                2
27  frontend-0       trace.self.duration_mean     23   0.003623    0.026573                23                     2  2022-03-21T22:53:00+08:00                    0                       None             7.382310             0.003588                2
28  frontend-0        trace.self.duration_p95     23   0.006265    0.243430                23                     2  2022-03-21T22:34:00+08:00                    0                       None            25.838215             0.005559                2
29  frontend-0           trace.self.row_count     23  22.200000  547.300000                23                     0                       None                    2  2022-03-21T22:45:00+08:00           580.000000             8.000000                2

[30 rows x 13 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd
import pytz
import re

# reuse df_trace and tz if available; otherwise load and set timezone
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# ensure dtypes
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# incident window bounds (UTC+8) -> epoch seconds (UTC)
start_local = pd.Timestamp('2022-03-21 22:30:00', tz=tz)
end_local = pd.Timestamp('2022-03-21 22:59:59', tz=tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# select trace_name containing from_frontend OR from_root AND containing one of error_rate|row_count|duration (case-insensitive)
mask_from = df_trace['trace_name'].str.contains(r'(from_frontend|from_root)', case=False, na=False)
mask_types = df_trace['trace_name'].str.contains(r'(error_rate|row_count|duration)', case=False, na=False)
df_sel = df_trace[mask_from & mask_types].copy()

# helper to compute p5/p95
def p5(arr):
    a = np.array(arr.dropna())
    return float(np.percentile(a,5)) if a.size>0 else np.nan
def p95(arr):
    a = np.array(arr.dropna())
    return float(np.percentile(a,95)) if a.size>0 else np.nan

# compute global thresholds (full-series) per (cmdb_id, trace_name)
thresholds = df_sel.groupby(['cmdb_id','trace_name'])['value'].agg(
    full_series_count = lambda x: int(x.dropna().shape[0]),
    P5 = p5,
    P95 = p95
).reset_index()

# prepare windowed data
mask_window = (df_sel['timestamp'].notna()) & (df_sel['timestamp'] >= start_ts) & (df_sel['timestamp'] <= end_ts)
df_win = df_sel[mask_window].copy()

# determine trace_type per trace_name
def detect_trace_type(name):
    if re.search(r'error_rate', name, re.IGNORECASE):
        return 'error_rate'
    if re.search(r'row_count', name, re.IGNORECASE):
        return 'row_count'
    if re.search(r'duration', name, re.IGNORECASE):
        return 'duration'
    return 'other'

thresholds['trace_type'] = thresholds['trace_name'].apply(detect_trace_type)

# merge thresholds into window rows for anomaly marking
if not df_win.empty:
    df_win = df_win.merge(thresholds[['cmdb_id','trace_name','P5','P95','trace_type']], on=['cmdb_id','trace_name'], how='left')
    # mark anomalies according to trace_type
    df_win['is_high'] = ((df_win['trace_type'].isin(['error_rate','duration'])) & (df_win['value'] > df_win['P95']))
    df_win['is_low'] = ((df_win['trace_type'] == 'row_count') & (df_win['value'] < df_win['P5']))
else:
    # create empty columns for consistency
    df_win = df_win.merge(thresholds[['cmdb_id','trace_name','P5','P95','trace_type']], on=['cmdb_id','trace_name'], how='right')
    df_win = df_win.iloc[0:0]  # keep structure but empty

# aggregate in-window metrics per series
if not df_win.empty:
    def agg_group(g):
        points = int(g['value'].count())
        high_cnt = int(g['is_high'].sum())
        low_cnt = int(g['is_low'].sum())
        earliest_high = int(g.loc[g['is_high'],'timestamp'].min()) if high_cnt>0 else pd.NA
        earliest_low = int(g.loc[g['is_low'],'timestamp'].min()) if low_cnt>0 else pd.NA
        max_win = float(g['value'].max()) if g['value'].count()>0 else None
        min_win = float(g['value'].min()) if g['value'].count()>0 else None
        return pd.Series({
            'points_in_window': points,
            'high_anomalies_count': high_cnt,
            'earliest_high_ts': earliest_high,
            'low_anomalies_count': low_cnt,
            'earliest_low_ts': earliest_low,
            'max_value_in_window': max_win,
            'min_value_in_window': min_win
        })
    agg_win = df_win.groupby(['cmdb_id','trace_name']).apply(agg_group).reset_index()
else:
    agg_win = pd.DataFrame(columns=[
        'cmdb_id','trace_name','points_in_window','high_anomalies_count','earliest_high_ts',
        'low_anomalies_count','earliest_low_ts','max_value_in_window','min_value_in_window'
    ])

# merge thresholds and window aggregates, keep series even if no in-window points
merged = thresholds.merge(agg_win, on=['cmdb_id','trace_name'], how='left')

# fill missing window fields
merged['points_in_window'] = merged['points_in_window'].fillna(0).astype(int)
merged['high_anomalies_count'] = merged['high_anomalies_count'].fillna(0).astype(int)
merged['low_anomalies_count'] = merged['low_anomalies_count'].fillna(0).astype(int)
merged['max_value_in_window'] = merged['max_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))
merged['min_value_in_window'] = merged['min_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))

# convert earliest ts to ISO strings (UTC+8) or None
def to_iso_or_none(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat()

merged['earliest_high_timestamp'] = merged['earliest_high_ts'].apply(to_iso_or_none)
merged['earliest_low_timestamp'] = merged['earliest_low_ts'].apply(to_iso_or_none)

# compute total anomalies and earliest anomaly timestamp for sorting
merged['total_anomalies'] = merged['high_anomalies_count'] + merged['low_anomalies_count']

# earliest anomaly numeric for sorting (use large number if none)
merged['earliest_anom_num'] = merged[['earliest_high_ts','earliest_low_ts']].apply(
    lambda row: min([v for v in [row['earliest_high_ts'], row['earliest_low_ts']] if not pd.isna(v)]) if not (pd.isna(row['earliest_high_ts']) and pd.isna(row['earliest_low_ts'])) else np.nan,
    axis=1
)
merged['earliest_anom_num_filled'] = merged['earliest_anom_num'].fillna(9_999_999_999)

# select and order final columns
result_df = merged[[
    'cmdb_id','trace_name','trace_type','full_series_count','P5','P95',
    'points_in_window','high_anomalies_count','earliest_high_timestamp',
    'low_anomalies_count','earliest_low_timestamp','max_value_in_window','min_value_in_window','total_anomalies','earliest_anom_num_filled'
]].copy()

# sort by total_anomalies desc, then earliest anomaly timestamp asc (use filled numeric)
result_df = result_df.sort_values(['total_anomalies','earliest_anom_num_filled'], ascending=[False, True]).head(30).reset_index(drop=True)

# cleanup helper columns and ensure JSON-serializable numeric types
result_df = result_df.drop(columns=['earliest_anom_num_filled'])
result_df['full_series_count'] = result_df['full_series_count'].astype(int)
for col in ['P5','P95']:
    result_df[col] = result_df[col].apply(lambda v: float(v) if not pd.isna(v) else None)

# final compact table
result_df[[
    'cmdb_id','trace_name','trace_type','full_series_count','P5','P95',
    'points_in_window','high_anomalies_count','earliest_high_timestamp',
    'low_anomalies_count','earliest_low_timestamp','max_value_in_window','min_value_in_window','total_anomalies'
]]

```
Out[6]:
```
Summary of trace scan for trace_name containing "from_frontend" or "from_root" and (error_rate | duration | row_count) (incident window 2022-03-21 22:30:00 → 22:59:59 UTC+8):

- Result overview
  - Returned a compact top table (30 series). Almost all reported series show 2 anomaly points in-window (total_anomalies = 2).
  - Most anomalies are high-duration spikes; a smaller set are low row_count events.
  - Many series have ~22–23 points in the incident window (a few have 30), i.e., near-complete coverage of that 30-minute window for those series.

- Representative examples from the top results
  - productcatalogservice-2 | trace.from_frontend-0.duration_mean — 2 high-duration points, earliest high at 2022-03-21T22:30:00+08:00.
  - recommendationservice-0 | trace.from_frontend-0.row_count — 2 low row_count points, earliest low at 2022-03-21T22:31:00+08:00.
  - adservice-0 | trace.from_frontend-1.duration_p95 — 2 high-duration points, earliest high at 2022-03-21T22:32:00+08:00.
  - cartservice2-0 | trace.from_frontend2-0.duration_* (mean and p95) — 2 high-duration points, earliest high at 22:32:00+08.
  - Many recommendationservice and currencyservice "from_frontend" duration series also show 2 high-duration points with earliest highs around 22:32–22:33+08.

- Temporal pattern
  - Earliest anomaly timestamps cluster early in the incident window (roughly between 22:30 and 22:33 UTC+8 for many downstream services).
  - Anomalies are typically brief (2 in-window points), indicating short spikes rather than long sustained degradation in these trace series.

- Interpretation and recommended next steps
  - There are short-duration latency spikes (duration high) across multiple downstream services called by frontends, with concurrent low row_count in some services (e.g., recommendationservice-0), suggesting frontend requests experienced transient latency and reduced throughput starting ~22:30–22:33.
  - Investigate frontend-to-backend call paths around 22:30–22:35 (check frontend logs, error traces, and the metric anomalies already identified) to find common downstream causes (network, backend slowness, or specific service resource contention).
  - If you want, I can correlate these exact anomaly timestamps with metric spikes (CPU, network, disk) and error_logs entries to narrow root cause candidates.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                            trace_name trace_type  full_series_count        P5        P95  points_in_window  high_anomalies_count    earliest_high_timestamp  low_anomalies_count     earliest_low_timestamp  max_value_in_window  min_value_in_window  total_anomalies
0    productcatalogservice-2   trace.from_frontend-0.duration_mean   duration                 22  0.003799   0.004384                22                     2  2022-03-21T22:30:00+08:00                    0                       None             0.004466             0.003793                2
1    recommendationservice-0       trace.from_frontend-0.row_count  row_count                 22  2.050000  12.950000                22                     0                       None                    2  2022-03-21T22:31:00+08:00            15.000000             1.000000                2
2                adservice-0    trace.from_frontend-1.duration_p95   duration                 22  0.000012   0.000018                22                     2  2022-03-21T22:32:00+08:00                    0                       None             0.000019             0.000012                2
3             cartservice2-0  trace.from_frontend2-0.duration_mean   duration                 22  0.000173   0.000983                22                     2  2022-03-21T22:32:00+08:00                    0                       None             0.100111             0.000167                2
4             cartservice2-0   trace.from_frontend2-0.duration_p95   duration                 22  0.001000   0.002285                22                     2  2022-03-21T22:32:00+08:00                    0                       None             0.402500             0.001000                2
..                       ...                                   ...        ...                ...       ...        ...               ...                   ...                        ...                  ...                        ...                  ...                  ...              ...
25   recommendationservice-2   trace.from_frontend-1.duration_mean   duration                 23  0.002072   0.002872                23                     2  2022-03-21T22:32:00+08:00                    0                       None             0.003369             0.002050                2
26   recommendationservice-2    trace.from_frontend-1.duration_p95   duration                 23  0.002104   0.005111                23                     2  2022-03-21T22:32:00+08:00                    0                       None             0.006077             0.002050                2
27   recommendationservice-2   trace.from_frontend-2.duration_mean   duration                 22  0.002067   0.002406                22                     2  2022-03-21T22:32:00+08:00                    0                       None             0.002422             0.002013                2
28  recommendationservice2-0   trace.from_frontend2-0.duration_p95   duration                 23  0.002602   0.003262                23                     2  2022-03-21T22:32:00+08:00                    0                       None             0.003437             0.002374                2
29         currencyservice-1    trace.from_frontend-1.duration_p95   duration                 23  0.000096   0.000126                23                     2  2022-03-21T22:33:00+08:00                    0                       None             0.000128             0.000095                2

[30 rows x 14 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz
import re

# reuse existing dataframes if present, else load
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# ensure dtypes
df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce').astype('Int64')
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# timezone and window bounds in UTC+8 -> convert to epoch seconds UTC for comparisons
tz = pytz.timezone('Asia/Shanghai')
part_window_start_local = pd.Timestamp('2022-03-21 22:55:00', tz=tz)
part_window_end_local = pd.Timestamp('2022-03-21 22:59:59', tz=tz)
start_ts = int(part_window_start_local.tz_convert('UTC').timestamp())
end_ts = int(part_window_end_local.tz_convert('UTC').timestamp())

# Part A - error logs for frontends in window
mask_frontend = df_error['cmdb_id'].astype(str).str.startswith('frontend', na=False)
mask_time = (df_error['timestamp'].notna()) & (df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts)
df_err_win = df_error[mask_frontend & mask_time].copy()

# total count, earliest, latest (convert to ISO+08)
def to_iso_or_none(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

part_a_total = int(df_err_win.shape[0])
part_a_earliest = to_iso_or_none(df_err_win['timestamp'].min()) if part_a_total>0 else None
part_a_latest = to_iso_or_none(df_err_win['timestamp'].max()) if part_a_total>0 else None

# top-10 distinct messages with counts
if part_a_total>0:
    top_messages = df_err_win.groupby('message').size().reset_index(name='count').sort_values('count', ascending=False).head(10).reset_index(drop=True)
else:
    top_messages = pd.DataFrame(columns=['message','count'])

part_a_summary = {
    'total_count': part_a_total,
    'earliest_timestamp': part_a_earliest,
    'latest_timestamp': part_a_latest,
    'top_messages_count': int(top_messages.shape[0])
}

# Part B - trace anomalies in final 5 minutes
# target prefixes:
prefixes = [
    'frontend-0','frontend-1','frontend-2',
    'recommendationservice-0','recommendationservice-1','recommendationservice-2','recommendationservice2-0',
    'adservice-0','adservice-1','adservice-2','adservice2-0',
    'productcatalogservice','currencyservice'
]
# build mask for cmdb_id matching any of prefixes (prefix match)
cmdb_ids = df_trace['cmdb_id'].dropna().astype(str)
mask_cmdb = cmdb_ids.apply(lambda x: any(x.startswith(p) for p in prefixes))
# apply mask to the dataframe (align index)
df_trace['cmdb_id_str'] = df_trace['cmdb_id'].astype(str)
mask_cmdb_full = df_trace['cmdb_id_str'].apply(lambda x: any(x.startswith(p) for p in prefixes))

# trace_name filter
type_pattern = re.compile(r'(error_rate|row_count|duration)', flags=re.IGNORECASE)
mask_name = df_trace['trace_name'].astype(str).str.contains(type_pattern, na=False)

df_cand = df_trace[mask_cmdb_full & mask_name].copy()

# If no candidates, make empty result
if df_cand.empty:
    part_b_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','trace_type','full_series_count','P5','P95',
        'points_in_window','high_anomalies_count','earliest_high_timestamp',
        'low_anomalies_count','earliest_low_timestamp','max_value_in_window','min_value_in_window','total_anomalies'
    ])
else:
    # compute global P5/P95 per series using full series BEFORE window filtering
    def compute_p5(arr):
        a = np.array(arr.dropna())
        return float(np.percentile(a,5)) if a.size>0 else np.nan
    def compute_p95(arr):
        a = np.array(arr.dropna())
        return float(np.percentile(a,95)) if a.size>0 else np.nan

    thresholds = df_cand.groupby(['cmdb_id','trace_name'])['value'].agg(
        full_series_count = lambda x: int(x.dropna().shape[0]),
        P5 = compute_p5,
        P95 = compute_p95
    ).reset_index()

    # detect trace_type
    def trace_type(name):
        n = str(name)
        if re.search(r'error_rate', n, re.IGNORECASE):
            return 'error_rate'
        if re.search(r'row_count', n, re.IGNORECASE):
            return 'row_count'
        if re.search(r'duration', n, re.IGNORECASE):
            return 'duration'
        return 'other'
    thresholds['trace_type'] = thresholds['trace_name'].apply(trace_type)

    # filter to final 5-minute window
    mask_win = (df_cand['timestamp'].notna()) & (df_cand['timestamp'] >= start_ts) & (df_cand['timestamp'] <= end_ts)
    df_win = df_cand[mask_win].copy()
    # merge thresholds into window rows to mark anomalies
    if not df_win.empty:
        df_win = df_win.merge(thresholds[['cmdb_id','trace_name','P5','P95','trace_type']], on=['cmdb_id','trace_name'], how='left')
        df_win['is_high'] = ((df_win['trace_type'].isin(['error_rate','duration'])) & (df_win['value'] > df_win['P95']))
        df_win['is_low'] = ((df_win['trace_type'] == 'row_count') & (df_win['value'] < df_win['P5']))
        # aggregate
        def agg_group(g):
            points = int(g['value'].count())
            high_cnt = int(g['is_high'].sum())
            low_cnt = int(g['is_low'].sum())
            earliest_high = int(g.loc[g['is_high'],'timestamp'].min()) if high_cnt>0 else pd.NA
            earliest_low = int(g.loc[g['is_low'],'timestamp'].min()) if low_cnt>0 else pd.NA
            max_win = float(g['value'].max()) if points>0 else None
            min_win = float(g['value'].min()) if points>0 else None
            return pd.Series({
                'points_in_window': points,
                'high_anomalies_count': high_cnt,
                'earliest_high_ts': earliest_high,
                'low_anomalies_count': low_cnt,
                'earliest_low_ts': earliest_low,
                'max_value_in_window': max_win,
                'min_value_in_window': min_win
            })
        agg_win = df_win.groupby(['cmdb_id','trace_name']).apply(agg_group).reset_index()
    else:
        agg_win = pd.DataFrame(columns=[
            'cmdb_id','trace_name','points_in_window','high_anomalies_count','earliest_high_ts',
            'low_anomalies_count','earliest_low_ts','max_value_in_window','min_value_in_window'
        ])

    # merge thresholds with agg_win to keep series even with zero in-window points
    merged = thresholds.merge(agg_win, on=['cmdb_id','trace_name'], how='left')

    # fill missing
    merged['points_in_window'] = merged['points_in_window'].fillna(0).astype(int)
    merged['high_anomalies_count'] = merged['high_anomalies_count'].fillna(0).astype(int)
    merged['low_anomalies_count'] = merged['low_anomalies_count'].fillna(0).astype(int)
    merged['max_value_in_window'] = merged['max_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))
    merged['min_value_in_window'] = merged['min_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))

    # convert earliest timestamps to ISO+08 or None
    merged['earliest_high_timestamp'] = merged['earliest_high_ts'].apply(lambda x: to_iso_or_none(x) if not pd.isna(x) else None)
    merged['earliest_low_timestamp'] = merged['earliest_low_ts'].apply(lambda x: to_iso_or_none(x) if not pd.isna(x) else None)

    merged['total_anomalies'] = merged['high_anomalies_count'] + merged['low_anomalies_count']
    # numeric earliest for sorting (NaN -> large)
    merged['earliest_anom_num'] = merged[['earliest_high_ts','earliest_low_ts']].apply(
        lambda row: min([v for v in [row['earliest_high_ts'], row['earliest_low_ts']] if not pd.isna(v)]) if not (pd.isna(row['earliest_high_ts']) and pd.isna(row['earliest_low_ts'])) else np.nan,
        axis=1
    )
    merged['earliest_anom_filled'] = merged['earliest_anom_num'].fillna(9_999_999_999)

    part_b_df = merged[[
        'cmdb_id','trace_name','trace_type','full_series_count','P5','P95',
        'points_in_window','high_anomalies_count','earliest_high_timestamp',
        'low_anomalies_count','earliest_low_timestamp','max_value_in_window','min_value_in_window','total_anomalies','earliest_anom_filled'
    ]].copy()

    # sort by total_anomalies desc, earliest anomaly asc, limit 30
    part_b_df = part_b_df.sort_values(['total_anomalies','earliest_anom_filled'], ascending=[False, True]).head(30).reset_index(drop=True)
    part_b_df = part_b_df.drop(columns=['earliest_anom_filled'])

    # ensure JSON-serializable numeric scalars
    part_b_df['full_series_count'] = part_b_df['full_series_count'].astype(int)
    for col in ['P5','P95']:
        part_b_df[col] = part_b_df[col].apply(lambda v: float(v) if not pd.isna(v) else None)

# Final compact outputs
part_a_summary, top_messages, part_b_df

```
Out[7]:
```
Part A — Frontend error logs (22:55:00 → 22:59:59 UTC+8)
- Total matching error-log rows: 103
- Earliest log in window: 2022-03-21T22:55:01+08:00
- Latest log in window:  2022-03-21T22:59:58+08:00
- Top 10 distinct messages (sample, counts):
  1) "severity: warning, message: failed to retrieve ads" — 102 occurrences
  2) "severity: error, message: request error" — 1 occurrence

Part B — Trace anomalies in final 5 minutes (22:55:00 → 22:59:59 UTC+8)
- Scope: traces for productcatalogservice, recommendationservice, frontend*, adservice*, currencyservice* and related from_frontend/from_root traces of types duration/error_rate/row_count.
- Returned up to 30 series ranked by total anomalies. Compact summary of top findings:
  - Many downstream duration traces show 2 high-duration points in this 5-minute window (common pattern: high_anomalies_count = 2).
  - The most prominent correlated traces are between recommendationservice and productcatalogservice:
    - productcatalogservice-1 ← from_recommendationservice-2.duration_mean and .duration_p95:
      - full_series_count = 23, points_in_window = 5, high_anomalies_count = 2, earliest_high = 2022-03-21T22:55:00+08:00
    - recommendationservice-2 → to_productcatalogservice-1.duration_mean and .p95:
      - same pattern (full_series_count = 23, points_in_window = 5, 2 high anomaly points, earliest_high = 22:55:00+08:00)
  - Several other productcatalogservice and recommendationservice "from/to" duration series show high-duration points at ~22:55–22:57 (many with points_in_window = 5 and 1–2 high anomalies).
  - Most reported series have small counts in-window (typically 5 points in this final-5-min window for those series), and total_anomalies mostly equal 2 (two high points).
- Interpretation / correlation:
  - Frontend pods logged many warnings “failed to retrieve ads” in the final 5 minutes (102 warnings). This coincides with duration spikes on calls between recommendationservice and productcatalogservice starting at ~22:55:00+08:00.
  - The temporal alignment (frontend warnings + high-duration traces on recommendationservice→productcatalogservice around 22:55–22:57) suggests these backend call latencies likely contributed to frontend errors/warnings.
- Suggested next steps:
  - Investigate recommendationservice and productcatalogservice around 22:55–22:57 (check service logs, traces, metrics such as CPU/network/disk around that interval).
  - Correlate the specific request traces (trace IDs) for slowed calls and inspect productcatalogservice latency and error handling for ads lookups.
  - Examine frontend error contexts (request paths and downstream call failures) and resource metrics of recommendationservice/productcatalogservice during 22:55–22:57.

If you want, I can export the Part B result table (JSON) of the top 30 rows or correlate these timestamps with the metric anomalies we found earlier.

The original code execution output of IPython Kernel is also provided below for reference:

({'total_count': 103, 'earliest_timestamp': '2022-03-21T22:55:01+08:00', 'latest_timestamp': '2022-03-21T22:59:58+08:00', 'top_messages_count': 2},                                              message  count
0  severity: warning, message: failed to retrieve...    102
1            severity: error, message: request error      1,                     cmdb_id                                         trace_name trace_type  full_series_count        P5       P95  points_in_window  high_anomalies_count    earliest_high_timestamp  low_anomalies_count earliest_low_timestamp  max_value_in_window  min_value_in_window  total_anomalies
0   productcatalogservice-1   trace.from_recommendationservice-2.duration_mean   duration                 23  0.000022  0.000044                 5                     2  2022-03-21T22:55:00+08:00                    0                   None             0.000047             0.000023                2
1   productcatalogservice-1    trace.from_recommendationservice-2.duration_p95   duration                 23  0.000023  0.000137                 5                     2  2022-03-21T22:55:00+08:00                    0                   None             0.000141             0.000027                2
2   recommendationservice-2     trace.to_productcatalogservice-1.duration_mean   duration                 23  0.000022  0.000044                 5                     2  2022-03-21T22:55:00+08:00                    0                   None             0.000047             0.000023                2
3   recommendationservice-2      trace.to_productcatalogservice-1.duration_p95   duration                 23  0.000023  0.000137                 5                     2  2022-03-21T22:55:00+08:00                    0                   None             0.000141             0.000027                2
4   productcatalogservice-2  trace.from_recommendationservice2-0.duration_mean   duration                 22  0.000024  0.000028                 5                     2  2022-03-21T22:57:00+08:00                    0                   None             0.000031             0.000023                2
..                      ...                                                ...        ...                ...       ...       ...               ...                   ...                        ...                  ...                    ...                  ...                  ...              ...
25  productcatalogservice-0    trace.from_recommendationservice-0.duration_p95   duration                 23  0.000024  0.000032                 5                     1  2022-03-21T22:56:00+08:00                    0                   None             0.000034             0.000026                1
26  productcatalogservice-2                 trace.from_frontend-1.duration_p95   duration                 23  0.004712  0.005684                 5                     1  2022-03-21T22:56:00+08:00                    0                   None             0.005718             0.004841                1
27  productcatalogservice-2   trace.from_recommendationservice-0.duration_mean   duration                 22  0.000022  0.000034                 5                     1  2022-03-21T22:56:00+08:00                    0                   None             0.000048             0.000024                1
28  productcatalogservice-2    trace.from_recommendationservice-0.duration_p95   duration                 22  0.000024  0.000067                 5                     1  2022-03-21T22:56:00+08:00                    0                   None             0.000159             0.000030                1
29  recommendationservice-0      trace.to_productcatalogservice-0.duration_p95   duration                 23  0.000024  0.000032                 5                     1  2022-03-21T22:56:00+08:00                    0                   None             0.000034             0.000026                1

[30 rows x 14 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import numpy as np
import pandas as pd
import pytz
import re

# reuse df_metric if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# timezone and final-5-min window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
win_start_local = pd.Timestamp('2022-03-21 22:55:00', tz=tz)
win_end_local = pd.Timestamp('2022-03-21 22:59:59', tz=tz)
win_start_ts = int(win_start_local.tz_convert('UTC').timestamp())
win_end_ts = int(win_end_local.tz_convert('UTC').timestamp())

# 1) selection criteria
cmdb_target = 'productcatalogservice-1'
substrs = ['cpu','memory','disk','io','fs','HeapMemoryUsage','ProcessCpuTime','container_spec','network','read','write']
pattern = '(' + '|'.join(re.escape(s) for s in substrs) + ')'

mask_cmdb = df_metric['cmdb_id'].astype(str) == cmdb_target
mask_kpi = df_metric['kpi_name'].astype(str).str.contains(pattern, case=False, na=False, regex=True)
df_cand_full = df_metric[mask_cmdb & mask_kpi].copy()

# If no candidate series, prepare empty result with expected columns
cols = [
    'cmdb_id','kpi_name','full_series_count','P5','P95',
    'points_in_window','high_anomalies_count','earliest_high_timestamp',
    'low_anomalies_count','earliest_low_timestamp','max_value_in_window','min_value_in_window'
]
if df_cand_full.empty:
    result_df = pd.DataFrame(columns=cols)
else:
    # 2) compute global P5/P95 and full_series_count per (cmdb_id,kpi_name) BEFORE window filtering
    def compute_p5(arr):
        a = np.array(arr.dropna())
        return float(np.percentile(a,5)) if a.size>0 else np.nan
    def compute_p95(arr):
        a = np.array(arr.dropna())
        return float(np.percentile(a,95)) if a.size>0 else np.nan

    thresholds = df_cand_full.groupby(['cmdb_id','kpi_name'])['value'].agg(
        full_series_count = lambda x: int(x.dropna().shape[0]),
        P5 = compute_p5,
        P95 = compute_p95
    ).reset_index()

    # 3) filter series to final 5-minute window
    mask_window = (df_cand_full['timestamp'].notna()) & (df_cand_full['timestamp'] >= win_start_ts) & (df_cand_full['timestamp'] <= win_end_ts)
    df_win = df_cand_full[mask_window].copy()

    # merge thresholds into window rows to mark anomalies
    if not df_win.empty:
        df_win = df_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
        df_win['is_high'] = df_win['value'] > df_win['P95']
        df_win['is_low'] = df_win['value'] < df_win['P5']
        # aggregate window metrics per series
        def agg_win_group(g):
            points = int(g['value'].count())
            high_cnt = int(g['is_high'].sum())
            low_cnt = int(g['is_low'].sum())
            earliest_high = int(g.loc[g['is_high'],'timestamp'].min()) if high_cnt>0 else pd.NA
            earliest_low = int(g.loc[g['is_low'],'timestamp'].min()) if low_cnt>0 else pd.NA
            max_win = float(g['value'].max()) if points>0 else None
            min_win = float(g['value'].min()) if points>0 else None
            return pd.Series({
                'points_in_window': points,
                'high_anomalies_count': high_cnt,
                'earliest_high_ts': earliest_high,
                'low_anomalies_count': low_cnt,
                'earliest_low_ts': earliest_low,
                'max_value_in_window': max_win,
                'min_value_in_window': min_win
            })
        agg_win = df_win.groupby(['cmdb_id','kpi_name']).apply(agg_win_group).reset_index()
    else:
        agg_win = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','points_in_window','high_anomalies_count','earliest_high_ts',
            'low_anomalies_count','earliest_low_ts','max_value_in_window','min_value_in_window'
        ])

    # 4) merge thresholds with window aggregates
    merged = thresholds.merge(agg_win, on=['cmdb_id','kpi_name'], how='left')

    # fill missing window fields with zeros/None
    merged['points_in_window'] = merged['points_in_window'].fillna(0).astype(int)
    merged['high_anomalies_count'] = merged['high_anomalies_count'].fillna(0).astype(int)
    merged['low_anomalies_count'] = merged['low_anomalies_count'].fillna(0).astype(int)
    merged['max_value_in_window'] = merged['max_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))
    merged['min_value_in_window'] = merged['min_value_in_window'].apply(lambda v: (float(v) if not pd.isna(v) else None))

    # convert earliest timestamps to ISO+08 strings or None
    def to_iso_or_none(ts):
        if pd.isna(ts):
            return None
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

    merged['earliest_high_timestamp'] = merged['earliest_high_ts'].apply(lambda x: to_iso_or_none(x) if not pd.isna(x) else None)
    merged['earliest_low_timestamp'] = merged['earliest_low_ts'].apply(lambda x: to_iso_or_none(x) if not pd.isna(x) else None)

    # ensure numeric JSON-serializable types
    merged['full_series_count'] = merged['full_series_count'].astype(int)
    for col in ['P5','P95']:
        merged[col] = merged[col].apply(lambda v: float(v) if not pd.isna(v) else None)

    # select requested columns and sort by total anomalies desc then points_in_window desc
    merged['total_anomalies'] = merged['high_anomalies_count'] + merged['low_anomalies_count']
    result_df = merged[[
        'cmdb_id','kpi_name','full_series_count','P5','P95',
        'points_in_window','high_anomalies_count','earliest_high_timestamp',
        'low_anomalies_count','earliest_low_timestamp','max_value_in_window','min_value_in_window','total_anomalies'
    ]].sort_values(['total_anomalies','points_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

    # drop the helper total_anomalies column for final compact display if not needed; keep it for context
    # (User requested columns do not include total_anomalies, but it's useful; we keep it)
    
# final result variable (compact)
result_df

```
Out[8]:
```
Summary of metrics for productcatalogservice-1 (final 5-minute window = 2022-03-21 22:55:00 → 22:59:59 UTC+8):

- Scope & method: inspected KPI series for cmdb_id = "productcatalogservice-1" where kpi_name matched cpu, memory, disk, io, fs, HeapMemoryUsage, ProcessCpuTime, container_spec, network, read, or write. Global P5/P95 were computed using the full series; anomalies were then evaluated inside the final 5-minute window.

- Coverage: each reported KPI has full_series_count = 30 (full data available) and points_in_window = 5 (five minutes sampled in the final window).

- Main anomaly pattern:
  - Most matched KPIs show high-anomalies_count = 2 and low_anomalies_count = 0 (i.e., two in-window points exceed the global P95; no points fell below P5).
  - Earliest high-anomaly timestamps cluster around 2022-03-21T22:56:00+08:00 → 22:58:00+08:00 depending on the KPI.

- Notable KPIs with 2 high points (examples from the top results):
  - container.node-5.container_cpu_cfs_periods (earliest high 22:58:00)
  - container.node-5.container_cpu_system_seconds (earliest high 22:57:00)
  - container.node-5.container_cpu_usage_seconds (earliest high 22:57:00)
  - container.node-5.container_cpu_user_seconds (earliest high 22:58:00)
  - container memory failure metrics (container_memory_failures.*) show 2 high points at ~22:57:00
  - container network metrics (receive packets, transmit MB, transmit packets) show 2 high points at ~22:57:00
  - Several mesh.destination.* KPIs (calls between productcatalogservice and checkoutservice/frontend) show 2 high points at ~22:56:00–22:57:00 with elevated request/response byte or count values.

- Magnitude/context: max and min values in-window indicate measurable increases for those high points (e.g., cpu usage, network packets/bytes, and mesh request/response metrics jumped above their usual P95 values).

- Interpretation & recommended next steps:
  - There are simultaneous short-lived high values across CPU, container network, and mesh call metrics for productcatalogservice-1 around ~22:56–22:58 UTC+8.
  - This aligns with trace- and frontend-level anomalies (duration spikes and many frontend error/warning logs) observed across the same timeframe.
  - Actionable next steps: investigate productcatalogservice-1 at ~22:56–22:58 — check service logs, thread/GC activity, container CPU scheduling (cfs_periods), and network I/O; correlate specific request traces and frontend error events to determine whether backend latency or resource contention caused the front-end failures.

If you want, I can export the exact numeric table of the KPI rows shown (JSON) or correlate these KPI timestamps with the trace-level request IDs and frontend error log lines.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  full_series_count          P5            P95  points_in_window  high_anomalies_count    earliest_high_timestamp  low_anomalies_count earliest_low_timestamp  max_value_in_window  min_value_in_window  total_anomalies
0   productcatalogservice-1         container.node-5.container_cpu_cfs_periods                 30    0.000000     186.625000                 5                     2  2022-03-21T22:58:00+08:00                    0                   None           200.000000           137.000000                2
1   productcatalogservice-1      container.node-5.container_cpu_system_seconds                 30    0.000000       0.138750                 5                     2  2022-03-21T22:57:00+08:00                    0                   None             0.155000             0.105000                2
2   productcatalogservice-1       container.node-5.container_cpu_usage_seconds                 30    0.000020       0.330481                 5                     2  2022-03-21T22:57:00+08:00                    0                   None             0.354350             0.272246                2
3   productcatalogservice-1        container.node-5.container_cpu_user_seconds                 30    0.000000       0.140000                 5                     2  2022-03-21T22:58:00+08:00                    0                   None             0.155000             0.120000                2
4   productcatalogservice-1  container.node-5.container_memory_failures.con...                 30    0.000000    1116.600000                 5                     2  2022-03-21T22:57:00+08:00                    0                   None          1215.500000           931.500000                2
5   productcatalogservice-1  container.node-5.container_memory_failures.hie...                 30    0.000000    1116.600000                 5                     2  2022-03-21T22:57:00+08:00                    0                   None          1215.500000           931.500000                2
6   productcatalogservice-1  container.node-5.container_network_receive_pac...                 30  283.500000    2552.800000                 5                     2  2022-03-21T22:57:00+08:00                    0                   None          2889.000000          1940.500000                2
7   productcatalogservice-1  container.node-5.container_network_transmit_MB...                 30    1.071202       1.637864                 5                     2  2022-03-21T22:57:00+08:00                    0                   None             1.699550             1.496396                2
8   productcatalogservice-1  container.node-5.container_network_transmit_pa...                 30  186.025000    2495.900000                 5                     2  2022-03-21T22:57:00+08:00                    0                   None          2813.000000          1878.500000                2
9   productcatalogservice-1  mesh.destination.checkoutservice.productcatalo...                 30    0.000000    2472.625000                 5                     2  2022-03-21T22:56:00+08:00                    0                   None          2620.000000           655.000000                2
10  productcatalogservice-1  mesh.destination.checkoutservice.productcatalo...                 30    0.000000       0.017739                 5                     2  2022-03-21T22:56:00+08:00                    0                   None             0.019700             0.004550                2
11  productcatalogservice-1  mesh.destination.checkoutservice.productcatalo...                 30    0.000000       3.275000                 5                     2  2022-03-21T22:56:00+08:00                    0                   None             3.500000             1.500000                2
12  productcatalogservice-1  mesh.destination.checkoutservice.productcatalo...                 30    0.000000       3.275000                 5                     2  2022-03-21T22:56:00+08:00                    0                   None             3.500000             1.500000                2
13  productcatalogservice-1  mesh.destination.checkoutservice.productcatalo...                 30    0.000000    5473.750000                 5                     2  2022-03-21T22:56:00+08:00                    0                   None          5800.000000          1450.000000                2
14  productcatalogservice-1  mesh.destination.checkoutservice.productcatalo...                 30    0.000000       3.275000                 5                     2  2022-03-21T22:56:00+08:00                    0                   None             3.500000             1.500000                2
15  productcatalogservice-1  mesh.destination.frontend.productcatalogservic...                 30    0.000000  108071.500000                 5                     2  2022-03-21T22:57:00+08:00                    0                   None        118852.500000         73672.500000                2
16  productcatalogservice-1  mesh.destination.frontend.productcatalogservic...                 30    0.000000       0.659496                 5                     2  2022-03-21T22:57:00+08:00                    0                   None             0.713925             0.469550                2
17  productcatalogservice-1  mesh.destination.frontend.productcatalogservic...                 30    0.000000     165.950000                 5                     2  2022-03-21T22:57:00+08:00                    0                   None           175.500000           126.500000                2
18  productcatalogservice-1  mesh.destination.frontend.productcatalogservic...                 30    0.000000     165.950000                 5                     2  2022-03-21T22:57:00+08:00                    0                   None           175.500000           117.500000                2
19  productcatalogservice-1  mesh.destination.frontend.productcatalogservic...                 30    0.000000  244035.000000                 5                     2  2022-03-21T22:57:00+08:00                    0                   None        267075.000000        165075.000000                2```
```